# EDA — Indices_MERGED.csv
## Analyse Exploratoire des Données — Indices Boursiers
### Bourse de Casablanca | 2022 — 2026

---
## 1. Importation des Bibliothèques

In [1]:
import pandas as pd
import numpy as np



---
## 2. Chargement des Donnees

In [2]:
df = pd.read_csv('D:/rouge_file/Analyse_du_performance_de_la_bourse_casablanca/Merged/Indices_MERGED.csv')
print(f'Dimensions        : {df.shape[0]:,} lignes x {df.shape[1]} colonnes')
print(f'Colonnes          : {list(df.columns)}')


Dimensions        : 22,005 lignes x 5 colonnes
Colonnes          : ['Type', 'Indices', 'Valeur Indice', 'Variation / veille', 'Variation / 31/12']


In [3]:
df.head(10)

,Type,Indices,Valeur Indice,Variation / veille,Variation / 31/12
0,Type,MASI,NaN,NaN,NaN
1,Non spécifié,MASI,"11 586,14","-0,23 %","-13,27 %"
2,Non spécifié,MASI (USD),"8 587,55","-0,23 %","-26,80 %"
3,Non spécifié,MASI (EUR),"11 102,87","-0,35 %","-15,11 %"
4,Non spécifié,MASI RENTABILITE BRUT,"34 870,41","-0,23 %","-10,33 %"
5,Non spécifié,MASI RENTABILITE NET,"30 713,45","-0,23 %","-10,78 %"
6,Morocco Stock Index 20,Morocco Stock Index 20,"934,89","-0,26 %","-13,89 %"
7,Casablanca ESG 10,Casablanca ESG 10,"865,28","-0,05 %","-13,30 %"
8,INDICES SECTORIELS,AGROALIMENTAIRE / PRODUCTION,"31 493,52","-1,57 %","-14,96 %"
9,INDICES SECTORIELS,ASSURANCES,"4 847,28","0,17 %","-12,22 %"


In [4]:
print('TYPES DE DONNEES')
print(df.dtypes)
print()


TYPES DE DONNEES
Type                  object
Indices               object
Valeur Indice         object
Variation / veille    object
Variation / 31/12     object
dtype: object



###  _Toutes les colonnes sont en type string.
###  __Les colonnes Valeur Indice et Variations necessitent une conversion en float.

---
## 3. Analyse de la Qualite des Donnees (Avant Nettoyage)

In [5]:
print(' VALEURS MANQUANTES ')
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Valeurs manquantes': missing, 'Pourcentage (%)': missing_pct})
print(missing_df)
print()


 VALEURS MANQUANTES 
                    Valeurs manquantes  Pourcentage (%)
Type                                 0             0.00
Indices                              0             0.00
Valeur Indice                      679             3.09
Variation / veille                 679             3.09
Variation / 31/12                  679             3.09



In [6]:
print(' DOUBLONS ')
nb_doublons = df.duplicated().sum()
print(f'Nombre de lignes dupliquees : {nb_doublons:,}')
print(f'Pourcentage                 : {nb_doublons / len(df) * 100:.2f}%')

 DOUBLONS 
Nombre de lignes dupliquees : 1,194
Pourcentage                 : 5.43%


In [7]:
print(' LIGNES PARASITES (Type = Type) ')
parasites = df[df['Type'] == 'Type']
print(f'Nombre de lignes parasites : {len(parasites):,}')
print()
print(parasites.head(3).to_string())
print()


 LIGNES PARASITES (Type = Type) 
Nombre de lignes parasites : 679

    Type Indices Valeur Indice Variation / veille Variation / 31/12
0   Type    MASI           NaN                NaN               NaN
31  Type   MASI®           NaN                NaN               NaN
63  Type   MASI®           NaN                NaN               NaN



In [21]:
#  Ce sont des en-tetes de tableau mal filtres lors de l extraction OCR.

In [8]:
print(' VALEURS UNIQUES PAR COLONNE ')
for col in df.columns:
    print(f'{col:30s} : {df[col].nunique()} valeurs uniques')

 VALEURS UNIQUES PAR COLONNE 
Type                           : 7 valeurs uniques
Indices                        : 59 valeurs uniques
Valeur Indice                  : 18103 valeurs uniques
Variation / veille             : 1254 valeurs uniques
Variation / 31/12              : 6884 valeurs uniques


In [9]:
print(' VALEURS UNIQUES — COLONNE TYPE ')
for val in df['Type'].unique():
    count = (df['Type'] == val).sum()
    print(f'  {str(val):45s} : {count:,} lignes')
print()


 VALEURS UNIQUES — COLONNE TYPE 
  Type                                          : 679 lignes
  Non spécifié                                  : 5,429 lignes
  Morocco Stock Index 20                        : 1 lignes
  Casablanca ESG 10                             : 1 lignes
  INDICES SECTORIELS                            : 23 lignes
  INDICES SECTORIELS MASI®                      : 15,803 lignes
  **INDICES SECTORIELS MASI®**                  : 69 lignes



In [ ]:
#3 variantes pour les indices sectoriels a harmoniser 
#  INDICES SECTORIELS / INDICES SECTORIELS MASI / **INDICES SECTORIELS MASI**

---
## 4. Nettoyage des Donnees

In [10]:
print(' ETAPE 1 : Suppression des lignes parasites ')
df_clean = df[df['Type'] != 'Type'].copy()
print(f'Lignes supprimees : {len(df) - len(df_clean):,}')
print(f'Shape apres       : {df_clean.shape}')

 ETAPE 1 : Suppression des lignes parasites 
Lignes supprimees : 679
Shape apres       : (21326, 5)


In [11]:
print(' ETAPE 2 : Suppression des doublons ')
avant    = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Doublons supprimes : {avant - len(df_clean):,}')
print(f'Shape apres        : {df_clean.shape}')

 ETAPE 2 : Suppression des doublons 
Doublons supprimes : 518
Shape apres        : (20808, 5)


In [12]:
print('ETAPE 3 : Normalisation colonne Type ')

mapping_type = {
    'Non specifie'                 : 'Indices Globaux',
    'Non spécifié'                 : 'Indices Globaux',
    'Morocco Stock Index 20'       : 'Morocco Stock Index 20',
    'Casablanca ESG 10'            : 'Casablanca ESG 10',
    'INDICES SECTORIELS'           : 'Indices Sectoriels',
    'INDICES SECTORIELS MASI\u00ae': 'Indices Sectoriels',
    '**INDICES SECTORIELS MASI\u00ae**': 'Indices Sectoriels',
}

df_clean['Type'] = df_clean['Type'].map(mapping_type).fillna(df_clean['Type'])

print('Valeurs apres normalisation :')
for val in df_clean['Type'].unique():
    count = (df_clean['Type'] == val).sum()
    print(f'  {str(val):40s} : {count:,} lignes')

ETAPE 3 : Normalisation colonne Type 
Valeurs apres normalisation :
  Indices Globaux                          : 5,429 lignes
  Morocco Stock Index 20                   : 1 lignes
  Casablanca ESG 10                        : 1 lignes
  Indices Sectoriels                       : 15,377 lignes


In [13]:
print(' ETAPE 4 : Conversion colonnes numeriques ')

def convertir_nombre(serie):
    return (
        serie
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .pipe(pd.to_numeric, errors='coerce')
    )

def convertir_pourcentage(serie):
    return (
        serie
        .str.replace('%', '', regex=False)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .pipe(pd.to_numeric, errors='coerce')
    )

df_clean['Valeur_Indice']      = convertir_nombre(df_clean['Valeur Indice'])
df_clean['Variation_Veille']   = convertir_pourcentage(df_clean['Variation / veille'])
df_clean['Variation_Annuelle'] = convertir_pourcentage(df_clean['Variation / 31/12'])

print('Types apres conversion :')
print(df_clean[['Valeur_Indice', 'Variation_Veille', 'Variation_Annuelle']].dtypes)
print()
print('Apercu :')
print(df_clean[['Indices', 'Valeur_Indice', 'Variation_Veille', 'Variation_Annuelle']].head(5).to_string())

 ETAPE 4 : Conversion colonnes numeriques 
Types apres conversion :
Valeur_Indice         float64
Variation_Veille      float64
Variation_Annuelle    float64
dtype: object

Apercu :
                 Indices  Valeur_Indice  Variation_Veille  Variation_Annuelle
1                   MASI       11586.14             -0.23              -13.27
2             MASI (USD)        8587.55             -0.23              -26.80
3             MASI (EUR)       11102.87             -0.35              -15.11
4  MASI RENTABILITE BRUT       34870.41             -0.23              -10.33
5   MASI RENTABILITE NET       30713.45             -0.23              -10.78


In [14]:
print(' RAPPORT FINAL DU NETTOYAGE ')
print(f'Lignes initiales          : {len(df):,}')
print(f'Lignes apres nettoyage    : {len(df_clean):,}')
print(f'Lignes supprimees (total) : {len(df) - len(df_clean):,}')
print(f'Taux de conservation      : {len(df_clean)/len(df)*100:.1f}%')
print()
print('Valeurs manquantes restantes :')
print(df_clean[['Valeur_Indice', 'Variation_Veille', 'Variation_Annuelle']].isnull().sum())

 RAPPORT FINAL DU NETTOYAGE 
Lignes initiales          : 22,005
Lignes apres nettoyage    : 20,808
Lignes supprimees (total) : 1,197
Taux de conservation      : 94.6%

Valeurs manquantes restantes :
Valeur_Indice         0
Variation_Veille      1
Variation_Annuelle    0
dtype: int64


---
## 5. Analyse Statistique Descriptive

In [15]:
print(' STATISTIQUES GLOBALES — VALEUR INDICE ')
stats = df_clean['Valeur_Indice'].describe()
print(stats)
print()
print(f'Mediane  : {df_clean["Valeur_Indice"].median():,.2f}')
print(f'Variance : {df_clean["Valeur_Indice"].var():,.2f}')
print(f'Skewness : {df_clean["Valeur_Indice"].skew():.4f}  (>0 = distribution asymetrique droite)')
print(f'Kurtosis : {df_clean["Valeur_Indice"].kurt():.4f}')

 STATISTIQUES GLOBALES — VALEUR INDICE 
count     20808.000000
mean      12843.428465
std       15012.187808
min          25.370000
25%        1724.140000
50%        6284.000000
75%       17283.120000
max      107719.840000
Name: Valeur_Indice, dtype: float64

Mediane  : 6,284.00
Variance : 225,365,782.79
Skewness : 1.6969  (>0 = distribution asymetrique droite)
Kurtosis : 2.7761


In [16]:
print(' STATISTIQUES PAR TYPE D INDICE ')
stats_type = df_clean.groupby('Type')['Valeur_Indice'].agg(
    Nb_lignes  = 'count',
    Moyenne    = 'mean',
    Min        = 'min',
    Max        = 'max',
    Ecart_type = 'std'
).round(2)
print(stats_type.to_string())

 STATISTIQUES PAR TYPE D INDICE 
                        Nb_lignes   Moyenne     Min        Max  Ecart_type
Type                                                                      
Casablanca ESG 10               1    865.28  865.28     865.28         NaN
Indices Globaux              5429  16781.91  772.83   66178.68    17714.51
Indices Sectoriels          15377  11454.46   25.37  107719.84    13666.19
Morocco Stock Index 20          1    934.89  934.89     934.89         NaN


In [17]:
print(' STATISTIQUES VARIATIONS ')
print()
print(' Variation / Veille (%) ')
print(df_clean['Variation_Veille'].describe().round(4))
print()
print(' Variation Annuelle (%) ')
print(df_clean['Variation_Annuelle'].describe().round(4))
print()
pos = (df_clean['Variation_Veille'] > 0).sum()
neg = (df_clean['Variation_Veille'] < 0).sum()
nul = (df_clean['Variation_Veille'] == 0).sum()
print(f'Seances positives : {pos:,} ({pos/len(df_clean)*100:.1f}%)')
print(f'Seances negatives : {neg:,} ({neg/len(df_clean)*100:.1f}%)')
print(f'Seances nulles    : {nul:,} ({nul/len(df_clean)*100:.1f}%)')

 STATISTIQUES VARIATIONS 

 Variation / Veille (%) 
count    20807.0000
mean         0.1068
std          1.6644
min        -10.0000
25%         -0.5400
50%          0.0000
75%          0.6600
max         10.0000
Name: Variation_Veille, dtype: float64

 Variation Annuelle (%) 
count    20808.0000
mean        18.9360
std         33.0803
min        -33.4500
25%          2.8475
50%         10.8900
75%         23.6100
max        370.0800
Name: Variation_Annuelle, dtype: float64

Seances positives : 10,158 (48.8%)
Seances negatives : 9,636 (46.3%)
Seances nulles    : 1,013 (4.9%)


---
## 6. Analyse par Indice

In [18]:
print(' LISTE DES INDICES DISPONIBLES ')
indices_count = df_clean.groupby('Indices')['Valeur_Indice'].count().sort_values(ascending=False)
print(f'Nombre total d indices : {df_clean["Indices"].nunique()}')
print()
print('Top 15 indices par nombre d observations :')
print(indices_count.head(15).to_string())

 LISTE DES INDICES DISPONIBLES 
Nombre total d indices : 57

Top 15 indices par nombre d observations :
Indices
MASI                                                  679
MASI (EUR)                                            679
MASI (USD)                                            679
MASI RENTABILITE BRUT                                 679
MASI RENTABILITE NET                                  679
MASI 20                                               678
MASI BATIMENT ET MATERIAUX DE CONSTRUCTION            678
MASI BANQUES                                          678
MASI AGROALIMENTAIRE / PRODUCTION                     678
MASI Mid and Small Cap                                678
MASI PARTICIPATION ET PROMOTION IMMOBILIERES          678
MASI MATERIELS,LOGICIELS ET SERVICES INFORMATIQUES    678
MASI MINES                                            678
MASI ESG                                              678
MASI DISTRIBUTEURS                                    678


In [19]:
print(' FOCUS MASI — Indice Principal ')
masi = df_clean[df_clean['Indices'] == 'MASI'].copy()
print(f'Nombre de seances MASI : {len(masi):,}')
print()
print(f'Valeur Min              : {masi["Valeur_Indice"].min():,.2f}')
print(f'Valeur Max              : {masi["Valeur_Indice"].max():,.2f}')
print(f'Valeur Moyenne          : {masi["Valeur_Indice"].mean():,.2f}')
print(f'Ecart-type              : {masi["Valeur_Indice"].std():,.2f}')
print()
print(f'Variation veille Moyenne : {masi["Variation_Veille"].mean():.4f}%')
print(f'Variation veille Max     : {masi["Variation_Veille"].max():.4f}%')
print(f'Variation veille Min     : {masi["Variation_Veille"].min():.4f}%')
print()
pos_masi = (masi['Variation_Veille'] > 0).sum()
neg_masi = (masi['Variation_Veille'] < 0).sum()
print(f'Seances haussieres : {pos_masi} ({pos_masi/len(masi)*100:.1f}%)')
print(f'Seances baissieres : {neg_masi} ({neg_masi/len(masi)*100:.1f}%)')

 FOCUS MASI — Indice Principal 
Nombre de seances MASI : 679

Valeur Min              : 10,390.66
Valeur Max              : 20,233.50
Valeur Moyenne          : 14,881.69
Ecart-type              : 2,922.36

Variation veille Moyenne : 0.0867%
Variation veille Max     : 5.0800%
Variation veille Min     : -5.6400%

Seances haussieres : 381 (56.1%)
Seances baissieres : 295 (43.4%)


In [20]:
print(' PERFORMANCE PAR INDICE SECTORIEL ')
sectoriels = df_clean[df_clean['Type'] == 'Indices Sectoriels'].copy()

perf_secteurs = sectoriels.groupby('Indices').agg(
    Nb_seances           = ('Valeur_Indice',    'count'),
    Valeur_Moyenne       = ('Valeur_Indice',    'mean'),
    Valeur_Min           = ('Valeur_Indice',    'min'),
    Valeur_Max           = ('Valeur_Indice',    'max'),
    Variation_Moy_Veille = ('Variation_Veille', 'mean'),
    Variation_Ann_Moy    = ('Variation_Annuelle','mean'),
).round(2).sort_values('Variation_Ann_Moy', ascending=False)

print('Secteurs classes par performance annuelle moyenne :')
print(perf_secteurs.to_string())

 PERFORMANCE PAR INDICE SECTORIEL 
Secteurs classes par performance annuelle moyenne :
                                                      Nb_seances  Valeur_Moyenne  Valeur_Min  Valeur_Max  Variation_Moy_Veille  Variation_Ann_Moy
Indices                                                                                                                                          
MASI PARTICIPATION ET PROMOTION IMMOBILIERES                 678        12726.46     2327.78    21966.08                  0.29              64.08
MASI INGENIERIES ET BIENS D'EQUIPEMENT INDUSTRIELS           625          169.37       81.98      486.80                  0.26              61.93
MASI SANTE                                                   663         3147.64      990.00     5576.93                  0.23              47.55
SYLVICULTURE & PAPIER                                          1           43.22       43.22       43.22                 -5.88              44.99
MASI LOISIRS ET HOTELS               

In [21]:
print(' TOP 5 MEILLEURS ET PIRES SECTEURS ')
print()
print('Top 5 meilleurs secteurs (variation annuelle moyenne) :')
print(perf_secteurs['Variation_Ann_Moy'].head(5).to_string())
print()
print('Top 5 pires secteurs :')
print(perf_secteurs['Variation_Ann_Moy'].tail(5).to_string())

 TOP 5 MEILLEURS ET PIRES SECTEURS 

Top 5 meilleurs secteurs (variation annuelle moyenne) :
Indices
MASI PARTICIPATION ET PROMOTION IMMOBILIERES          64.08
MASI INGENIERIES ET BIENS D'EQUIPEMENT INDUSTRIELS    61.93
MASI SANTE                                            47.55
SYLVICULTURE & PAPIER                                 44.99
MASI LOISIRS ET HOTELS                                41.36

Top 5 pires secteurs :
Indices
TRANSPORT                              -14.40
AGROALIMENTAIRE / PRODUCTION           -14.96
TELECOMMUNICATIONS                     -17.89
SOCIETES DE PORTEFEUILLES - HOLDINGS   -18.08
BATIMENT & MATERIAUX DE CONSTRUCTION   -23.93


---
## 7. Analyse des Valeurs Aberrantes (Outliers)

In [22]:
print(' DETECTION DES OUTLIERS — Methode IQR ')
print()

for col in ['Valeur_Indice', 'Variation_Veille', 'Variation_Annuelle']:
    Q1       = df_clean[col].quantile(0.25)
    Q3       = df_clean[col].quantile(0.75)
    IQR      = Q3 - Q1
    lower    = Q1 - 1.5 * IQR
    upper    = Q3 + 1.5 * IQR
    outliers = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)]
    print(f'{col}:')
    print(f'  Q1={Q1:.2f} | Q3={Q3:.2f} | IQR={IQR:.2f}')
    print(f'  Borne inferieure={lower:.2f} | Borne superieure={upper:.2f}')
    print(f'  Outliers detectes : {len(outliers):,} ({len(outliers)/len(df_clean)*100:.2f}%)')
    print()

 DETECTION DES OUTLIERS — Methode IQR 

Valeur_Indice:
  Q1=1724.14 | Q3=17283.12 | IQR=15558.98
  Borne inferieure=-21614.33 | Borne superieure=40621.59
  Outliers detectes : 1,478 (7.10%)

Variation_Veille:
  Q1=-0.54 | Q3=0.66 | IQR=1.20
  Borne inferieure=-2.34 | Borne superieure=2.46
  Outliers detectes : 2,371 (11.39%)

Variation_Annuelle:
  Q1=2.85 | Q3=23.61 | IQR=20.76
  Borne inferieure=-28.30 | Borne superieure=54.75
  Outliers detectes : 1,770 (8.51%)



In [23]:
print(' VARIATIONS EXTREMES — Variation / Veille ')
print()
print('Top 10 plus fortes hausses :')
top_hausses = df_clean.nlargest(10, 'Variation_Veille')[['Indices', 'Valeur_Indice', 'Variation_Veille']]
print(top_hausses.to_string(index=False))
print()
print('Top 10 plus fortes baisses :')
top_baisses = df_clean.nsmallest(10, 'Variation_Veille')[['Indices', 'Valeur_Indice', 'Variation_Veille']]
print(top_baisses.to_string(index=False))

 VARIATIONS EXTREMES — Variation / Veille 

Top 10 plus fortes hausses :
                Indices  Valeur_Indice  Variation_Veille
         MASI TRANSPORT        4075.87             10.00
MASI TELECOMMUNICATIONS        1454.49              9.99
         MASI TRANSPORT        3368.92              9.99
         MASI TRANSPORT        3705.44              9.99
         MASI TRANSPORT        3687.88              9.99
         MASI TRANSPORT        4056.26              9.99
MASI INDUSTRIE AGRICOLE        2199.75              9.99
         MASI TRANSPORT        3660.93              9.99
MASI INDUSTRIE AGRICOLE        1209.50              9.98
MASI INDUSTRIE AGRICOLE        1330.25              9.98

Top 10 plus fortes baisses :
                                     Indices  Valeur_Indice  Variation_Veille
                 MASI SYLVICULTURE ET PAPIER          41.48            -10.00
                     MASI TELECOMMUNICATIONS        1373.91             -9.99
                              MASI T

---
## 8. Analyse de Correlation

In [24]:
print(' MATRICE DE CORRELATION ')
corr = df_clean[['Valeur_Indice', 'Variation_Veille', 'Variation_Annuelle']].corr().round(4)
print(corr.to_string())
print()
print('Interpretation :')
v1 = corr.loc['Valeur_Indice', 'Variation_Veille']
v2 = corr.loc['Valeur_Indice', 'Variation_Annuelle']
v3 = corr.loc['Variation_Veille', 'Variation_Annuelle']
print(f'  Valeur vs Variation/Veille   : {v1:.4f}')
print(f'  Valeur vs Variation/Annuelle : {v2:.4f}')
print(f'  Var Veille vs Var Annuelle   : {v3:.4f}')

 MATRICE DE CORRELATION 
                    Valeur_Indice  Variation_Veille  Variation_Annuelle
Valeur_Indice              1.0000            0.0014              0.0191
Variation_Veille           0.0014            1.0000              0.0643
Variation_Annuelle         0.0191            0.0643              1.0000

Interpretation :
  Valeur vs Variation/Veille   : 0.0014
  Valeur vs Variation/Annuelle : 0.0191
  Var Veille vs Var Annuelle   : 0.0643


---
## 9. Resume & Conclusions

In [27]:
top_sect   = perf_secteurs['Variation_Ann_Moy'].idxmax()
worst_sect = perf_secteurs['Variation_Ann_Moy'].idxmin()

print('RESUME EDA — Indices_MERGED.csv')
print()
print('DONNEES BRUTES :')
print(f'  Lignes initiales : 22 005   |   Colonnes : 5')
print()
print('PROBLEMES DETECTES :')
print(f'  Lignes parasites             : 679')
print(f'  Doublons                     : 1 194')
print(f'  Colonnes numeriques en string: 3')
print(f'  Variantes colonne Type       : 7 -> harmonisees en 4')
print()
print('APRES NETTOYAGE :')
print(f'  Lignes propres   : {len(df_clean):,}')
print(f'  Indices uniques  : {df_clean["Indices"].nunique()}')
print(f'  Types d indices  : {df_clean["Type"].nunique()}')
print()
print('INSIGHTS CLES :')
print(f'  MASI Min          : {df_clean[df_clean["Indices"]=="MASI"]["Valeur_Indice"].min():,.2f}')
print(f'  MASI Max          : {df_clean[df_clean["Indices"]=="MASI"]["Valeur_Indice"].max():,.2f}')
print(f'  Meilleur secteur  : {top_sect}')
print(f'  Pire secteur      : {worst_sect}')
print()
print('PROCHAINE ETAPE : Ajout colonne Date_Seance + chargement PostgreSQL')


RESUME EDA — Indices_MERGED.csv

DONNEES BRUTES :
  Lignes initiales : 22 005   |   Colonnes : 5

PROBLEMES DETECTES :
  Lignes parasites             : 679
  Doublons                     : 1 194
  Colonnes numeriques en string: 3
  Variantes colonne Type       : 7 -> harmonisees en 4

APRES NETTOYAGE :
  Lignes propres   : 20,808
  Indices uniques  : 57
  Types d indices  : 4

INSIGHTS CLES :
  MASI Min          : 10,390.66
  MASI Max          : 20,233.50
  Meilleur secteur  : MASI PARTICIPATION ET PROMOTION IMMOBILIERES
  Pire secteur      : BATIMENT & MATERIAUX DE CONSTRUCTION

PROCHAINE ETAPE : Ajout colonne Date_Seance + chargement PostgreSQL


---
## 10. Export du Dataset Nettoye

In [28]:
df_final = df_clean[['Type', 'Indices', 'Valeur_Indice', 'Variation_Veille', 'Variation_Annuelle']].copy()

df_final.to_csv('Indices_CLEAN.csv', index=False, encoding='utf-8')

print('Dataset nettoye exporte : Indices_CLEAN.csv')
print(f'Shape finale : {df_final.shape}')
print()
df_final.head(10)

Dataset nettoye exporte : Indices_CLEAN.csv
Shape finale : (20808, 5)



,Type,Indices,Valeur_Indice,Variation_Veille,Variation_Annuelle
1,Indices Globaux,MASI,11586.14,-0.23,-13.27
2,Indices Globaux,MASI (USD),8587.55,-0.23,-26.80
3,Indices Globaux,MASI (EUR),11102.87,-0.35,-15.11
4,Indices Globaux,MASI RENTABILITE BRUT,34870.41,-0.23,-10.33
5,Indices Globaux,MASI RENTABILITE NET,30713.45,-0.23,-10.78
6,Morocco Stock Index 20,Morocco Stock Index 20,934.89,-0.26,-13.89
7,Casablanca ESG 10,Casablanca ESG 10,865.28,-0.05,-13.30
8,Indices Sectoriels,AGROALIMENTAIRE / PRODUCTION,31493.52,-1.57,-14.96
9,Indices Sectoriels,ASSURANCES,4847.28,0.17,-12.22
10,Indices Sectoriels,BANQUES,12136.89,-1.14,-13.96
